In [ ]:
import numpy as np

from scipy.fft import fft, ifft, dct, idct, dst, idst

from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, QFTGate, UnitaryGate
from qiskit.quantum_info import Statevector

In [ ]:
WINDOW_SIZES = [256] #2^8 must n^2: 64, 128, 256, 512

def validate_window(x):
    x = np.asarray(x)
    if x.ndim != 1:
        raise ValueError("Transform input must be 1-D")

    n = len(x)
    if n == 0 or (n(n-1)) != 0:
        raise ValueError(f"Window length must be a power of two, got {n}")

    return x 

# Classical FFT (Fast Fourier Transforms)

In [ ]:
def classical_fft(x):
    x = validate_window(x)
    return fft(x, norm="ortho")

def classical_ifft(coeffs):
    coeffs = validate_window(coeffs)
    return ifft(coeffs, norm="ortho")

# Classical DCT-II (Discrete Cosine Transforms II)

In [ ]:
def classical_dct(x):
    x = validate_window(x)
    return dct(x, type=2, norm="ortho")

def classical_idct(coeffs):
    coeffs = validate_window(coeffs)
    return idct(coeffs, type=2, norm="ortho")

# Classical DST-II (Discrete Sine Transforms II)

In [ ]:
def classical_dst(x):
    x = validate_window(x)
    return dst(x, type=2, norm="ortho")


def classical_idst(coeffs):
    coeffs = validate_window(coeffs)
    return idst(coeffs, type=2, norm="ortho")

# Classical DWT-Haar (Discrete Wavelet Transforms Haar)

In [ ]:
def classical_haar(x):
    x = validate_window(x).astype(np.float64)
    approx = x.copy()
    details = []
    while len(approx) > 1:
        even = approx[0::2]
        odd = approx[1::2]
        next_approx = (even + odd) / np.sqrt(2.0)
        detail = (even - odd) / np.sqrt(2.0)
        details.append(detail)
        approx = next_approx

    return np.concatenate([approx, *details[::-1]])

def classical_ihaar(coeffs):
    coeffs = validate_window(coeffs).astype(np.float64)
    approx = coeffs[:1]
    offset = 1
    while offset < len(coeffs):
        n = len(approx)
        detail = coeffs[offset:offset+n]
        reconstructed = np.empty(2 * n, dtype=np.float64)
        reconstructed[0::2] = (approx + detail) / np.sqrt(2.0)
        reconstructed[1::2] = (approx - detail) / np.sqrt(2.0)
        approx = reconstructed
        offset += n

    return approx